# Chapitre 6 - Cours 1 : découper les données, par plage de clés ou par hachage

Quand le volume de données ou le débit de requêtes dépasse ce qu'une machine peut
absorber, on découpe le jeu de données en morceaux, les **partitions**, et l'on place des
partitions différentes sur des nœuds différents. Chaque enregistrement appartient à une
seule partition ; chaque partition est en quelque sorte une petite base autonome. Le but
premier est la **scalabilité** : répartir données et requêtes sur de nombreux disques et
processeurs. Ce notebook couvre l'introduction du chapitre (avec l'encadré
*Terminological confusion*), la section *Partitioning and Replication* et la section
*Partitioning of Key-Value Data* jusqu'aux sous-sections *Partitioning by Key Range* et
*Partitioning by Hash of Key* (le rééquilibrage fait l'objet du cours 2).

**Plan.**

1. Partitionnement et réplication : deux mécanismes indépendants qui se combinent.
2. L'objectif : une répartition équitable, et ce que coûte le déséquilibre.
3. Le partitionnement par plage de clés : l'encyclopédie en volumes.
4. Le piège des horodatages : un point chaud qui se déplace avec le temps.
5. Le partitionnement par hachage : uniformité retrouvée, requêtes de plage perdues.

In [1]:
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import partitioning as P
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET, COLORS,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
rng = np.random.default_rng(0)
fmt = lambda v: f"{v:,.0f}".replace(",", " ")      # 1234567 -> « 1 234 567 »